## Readme

This code is designed to decode images acquired from five fluorescence channels across three imaging rounds, expanding the theoretical pool of available barcodes to 125 (5<sup>3</sup>). To run this code, the environmental requirements are identical to those employed in the main code. The decoding simulation results are presented in Supplementary Fig. 12 and Supplementary Note 5. The original image and decoding data are deposited in the "Decoding simulation for five fluorescence channels" folder on GitHub, which includes:
- 1_Shared data for all crosstalk groups
- 2_Protein decoding with Zeiss crosstalk
- 3_Protein decoding with 1% crosstalk
- 4_Protein decoding with 5% crosstalk

## Step 1: Extra fluoro-changing information as one-dimensional array for each pixel

For each pixel in the FcTags-based IF image, the color-changing information over different time points is stored in a one-dimensional array_m and displayed as follows: 
```
array_m = [T1_F1, T1_F2, T1_F3, T1_F4, T1_F5, T2_F1, T2_F2, T2_F3, T2_F4, T2_F5, T3_F1, T3_F2, T3_F3, T3_F4, T3_F5]
```

- `T1_F1`, `T1_F2`, `T1_F3`, `T1_F4`, `T1_F5`: Five fluorescence channels obtained at round 1
- `T2_F1`, `T2_F2`, `T2_F3`, `T2_F4`, `T2_F5`: Five fluorescence channels obtained at round 2
- `T3_F1`, `T3_F2`, `T3_F3`, `T3_F4`, `T3_F5`: Five fluorescence channels obtained at round 3

The color-changing information for each pixel is then stored in a worksheet, where the cell coordinates correspond to the pixel coordinates in the image.

In [2]:
import os
import pandas as pd
from PIL import Image
from openpyxl import Workbook

# image file path
image_files = [
    'D://decoding simulation//T1_F1.tif',
    'D://decoding simulation//T1_F2.tif',
    'D://decoding simulation//T1_F3.tif',
    'D://decoding simulation//T1_F4.tif',
    'D://decoding simulation//T1_F5.tif',
    'D://decoding simulation//T2_F1.tif',
    'D://decoding simulation//T2_F2.tif',
    'D://decoding simulation//T2_F3.tif',
    'D://decoding simulation//T2_F4.tif',
    'D://decoding simulation//T2_F5.tif',
    'D://decoding simulation//T3_F1.tif',
    'D://decoding simulation//T3_F2.tif',
    'D://decoding simulation//T3_F3.tif',
    'D://decoding simulation//T3_F4.tif',
    'D://decoding simulation//T3_F5.tif'
]

# create a new workbook
wb = Workbook()
ws = wb.active
ws.title = "Gray Values"

# create a dictionary to store the grayscale values of coordinates
gray_value_dict = {}

# iterate through each image file
for image_file in image_files:
    with Image.open(image_file) as img:
        
        # get image size
        width, height = img.size
        
        # extract the grayscale value of each pixel
        for y in range(height):
            for x in range(width):
                value = img.getpixel((x, y))
                
                # use the coordinates (x, y) as the key in the dictionary
                if (x, y) not in gray_value_dict:
                    gray_value_dict[(x, y)] = []
                gray_value_dict[(x, y)].append(value)

# write the grayscale values to the worksheet
for (x, y), values in gray_value_dict.items():
    # Combine grayscale values with the same coordinates into a one-dimensional array
    combined_values = ', '.join(map(str, values))
    ws.cell(row=y + 1, column=x + 1, value=combined_values)

# write the grayscale values to the worksheet
output_file = 'D://decoding simulation//fluoro_changing_data.xlsx'
wb.save(output_file)

print(f"grayscale values have been exported to {output_file}")

grayscale values have been exported to D://decoding simulation//fluoro_changing_data.xlsx


## Step 2: Synthesize crosstalk data

Notes: Images with simulated crosstalk cannot be directly generated in image format, as the grayscale values in images are integers—making it impossible to accurately record the decimal values associated with crosstalk. Given that subsequent protein decoding is performed in Excel, we directly used Excel to generate crosstalk data to preserve all decimal places.

### Step 2.1: Extract gray values from .tif images (no crosstalk) and save them to .xls files

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob

# Set the directory where images are located
image_dir = "D://decoding simulation"

# Get all image files in the directory (supports common image formats)
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob(os.path.join(image_dir, ext)))

# If no images are found
if not image_paths:
    print(f"No image files found in directory {image_dir}")
else:
    # Process each image
    for img_path in image_paths:
        # Read the image (in grayscale mode)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            print(f"Unable to read image: {img_path}")
            continue
        
        # Get the image filename (without extension) for the Excel filename
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        excel_path = os.path.join(image_dir, f"{img_name}_grayscale.xlsx")
        
        # Convert the grayscale value array to DataFrame (no header)
        df = pd.DataFrame(img)
        
        # Save to Excel without index and header
        df.to_excel(excel_path, index=False, header=False)
        
        print(f"Processed image: {img_path}")
        print(f"Grayscale values saved to: {excel_path}\n")

print("All images processed")

### Step 2.2: Add crosstalk to the extracted .xls files as Zeiss settings

In [ ]:
import pandas as pd
import os

# Set the folder path
folder_path = "D://decoding simulation"

# Define dataset prefixes (T1, T2, T3) and channels included in each dataset
prefixes = ["T1", "T2", "T3"]
channels = ["AF405", "AF488", "Cy3", "Cy5", "Cy7"]

# Read all Excel file data
data = {}
for prefix in prefixes:
    for channel in channels:
        filename = f"{prefix}_{channel}.xlsx"
        file_path = os.path.join(folder_path, filename)
        try:
            # Read Excel file without header
            df = pd.read_excel(file_path, header=None)
            data[f"{prefix}_{channel}"] = df
            print(f"Successfully read file: {filename}")
        except Exception as e:
            print(f"Error reading file {filename}: {str(e)}")

# Define calculation formula templates (using {prefix} placeholder)
calculations = [
    {
        "output": "{prefix}_AF405_crosstalk",
        "formula": lambda p, d: d[f"{p}_AF405"] * 1 + d[f"{p}_AF488"] * 0.02
    },
    {
        "output": "{prefix}_AF488_crosstalk",
        "formula": lambda p, d: d[f"{p}_AF488"] * 0.98
    },
    {
        "output": "{prefix}_Cy3_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy3"] * 1
    },
    {
        "output": "{prefix}_Cy5_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy5"] * 1 + d[f"{p}_Cy7"] * 0.02
    },
    {
        "output": "{prefix}_Cy7_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy7"] * 0.98
    }
]

# Check if all files were successfully read
missing_files = []
for prefix in prefixes:
    for channel in channels:
        key = f"{prefix}_{channel}"
        if key not in data:
            missing_files.append(f"{prefix}_{channel}.xlsx")

if missing_files:
    print("The following required files are missing, cannot complete calculations:")
    for file in missing_files:
        print(f"- {file}")
else:
    # Batch calculate and save results by prefix
    for prefix in prefixes:
        print(f"\nStarting processing {prefix} dataset...")
        for calc in calculations:
            output_name = calc["output"].format(prefix=prefix)
            try:
                # Apply calculation formula
                result_df = calc["formula"](prefix, data)
                
                # Save to new Excel file
                output_path = os.path.join(folder_path, f"{output_name}.xlsx")
                result_df.to_excel(output_path, index=False, header=False)
                print(f"Generated: {output_name}.xlsx")
            except Exception as e:
                print(f"Error calculating {output_name}: {str(e)}")
    
    print("\nAll datasets processed")

### Step 2.3: Add 1% crosstalk to the extracted .xls files

In [ ]:
import pandas as pd
import os

# Set the folder path
folder_path = "D://decoding simulation"

# Define dataset prefixes (T1, T2, T3) and channels included in each dataset
prefixes = ["T1", "T2", "T3"]
channels = ["AF405", "AF488", "Cy3", "Cy5", "Cy7"]

# Read all Excel file data
data = {}
for prefix in prefixes:
    for channel in channels:
        filename = f"{prefix}_{channel}.xlsx"
        file_path = os.path.join(folder_path, filename)
        try:
            # Read Excel file without header
            df = pd.read_excel(file_path, header=None)
            data[f"{prefix}_{channel}"] = df
            print(f"Successfully read file: {filename}")
        except Exception as e:
            print(f"Error reading file {filename}: {str(e)}")

# Define calculation formula templates (using {prefix} placeholder)
calculations = [
    {
        "output": "{prefix}_AF405_crosstalk",
        "formula": lambda p, d: d[f"{p}_AF405"] * 0.99 + d[f"{p}_AF488"] * 0.01
    },
    {
        "output": "{prefix}_AF488_crosstalk",
        "formula": lambda p, d: d[f"{p}_AF488"] * 0.99 + d[f"{p}_AF405"] * 0.01 + d[f"{p}_Cy3"] * 0.01
    },
    {
        "output": "{prefix}_Cy3_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy3"] * 0.99 + d[f"{p}_AF488"] * 0.01 + d[f"{p}_Cy5"] * 0.01
    },
    {
        "output": "{prefix}_Cy5_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy5"] * 0.99 + d[f"{p}_Cy3"] * 0.01 + d[f"{p}_Cy7"] * 0.01
    },
    {
        "output": "{prefix}_Cy7_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy7"] * 0.99 + d[f"{p}_Cy5"] * 0.01
    }
]

# Check if all files were successfully read
missing_files = []
for prefix in prefixes:
    for channel in channels:
        key = f"{prefix}_{channel}"
        if key not in data:
            missing_files.append(f"{prefix}_{channel}.xlsx")

if missing_files:
    print("The following required files are missing, cannot complete calculations:")
    for file in missing_files:
        print(f"- {file}")
else:
    # Batch calculate and save results by prefix
    for prefix in prefixes:
        print(f"\nStarting processing {prefix} dataset...")
        for calc in calculations:
            output_name = calc["output"].format(prefix=prefix)
            try:
                # Apply calculation formula
                result_df = calc["formula"](prefix, data)
                
                # Save to new Excel file
                output_path = os.path.join(folder_path, f"{output_name}.xlsx")
                result_df.to_excel(output_path, index=False, header=False)
                print(f"Generated: {output_name}.xlsx")
            except Exception as e:
                print(f"Error calculating {output_name}: {str(e)}")
    
    print("\nAll datasets processed")

### Additional step 2.4: Add 5% crosstalk to the extracted .xls files

In [ ]:
import pandas as pd
import os

# Set the folder path
folder_path = "D://decoding simulation"

# Define dataset prefixes (T1, T2, T3) and channels included in each dataset
prefixes = ["T1", "T2", "T3"]
channels = ["AF405", "AF488", "Cy3", "Cy5", "Cy7"]

# Read all Excel file data
data = {}
for prefix in prefixes:
    for channel in channels:
        filename = f"{prefix}_{channel}.xlsx"
        file_path = os.path.join(folder_path, filename)
        try:
            # Read Excel file without header
            df = pd.read_excel(file_path, header=None)
            data[f"{prefix}_{channel}"] = df
            print(f"Successfully read file: {filename}")
        except Exception as e:
            print(f"Error reading file {filename}: {str(e)}")

# Define calculation formula templates (using {prefix} placeholder)
calculations = [
    {
        "output": "{prefix}_AF405_crosstalk",
        "formula": lambda p, d: d[f"{p}_AF405"] * 0.95 + d[f"{p}_AF488"] * 0.05
    },
    {
        "output": "{prefix}_AF488_crosstalk",
        "formula": lambda p, d: d[f"{p}_AF488"] * 0.90 + d[f"{p}_AF405"] * 0.05 + d[f"{p}_Cy3"] * 0.05
    },
    {
        "output": "{prefix}_Cy3_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy3"] * 0.90 + d[f"{p}_AF488"] * 0.05 + d[f"{p}_Cy5"] * 0.05
    },
    {
        "output": "{prefix}_Cy5_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy5"] * 0.90 + d[f"{p}_Cy3"] * 0.05 + d[f"{p}_Cy7"] * 0.05
    },
    {
        "output": "{prefix}_Cy7_crosstalk",
        "formula": lambda p, d: d[f"{p}_Cy7"] * 0.95 + d[f"{p}_Cy5"] * 0.05
    }
]

# Check if all files were successfully read
missing_files = []
for prefix in prefixes:
    for channel in channels:
        key = f"{prefix}_{channel}"
        if key not in data:
            missing_files.append(f"{prefix}_{channel}.xlsx")

if missing_files:
    print("The following required files are missing, cannot complete calculations:")
    for file in missing_files:
        print(f"- {file}")
else:
    # Batch calculate and save results by prefix
    for prefix in prefixes:
        print(f"\nStarting processing {prefix} dataset...")
        for calc in calculations:
            output_name = calc["output"].format(prefix=prefix)
            try:
                # Apply calculation formula
                result_df = calc["formula"](prefix, data)
                
                # Save to new Excel file
                output_path = os.path.join(folder_path, f"{output_name}.xlsx")
                result_df.to_excel(output_path, index=False, header=False)
                print(f"Generated: {output_name}.xlsx")
            except Exception as e:
                print(f"Error calculating {output_name}: {str(e)}")
    
    print("\nAll datasets processed")

### Step 2.5: Synthesize fluoro_changing_data.xlsx based on the excels with crosstalk

In [11]:
import pandas as pd
import os

# Set the folder path
folder_path = "D://decoding simulation"

# Define the order of file names (arranged in the order of 15-digit numbers)
file_order = [
    "T1_AF405_crosstalk",
    "T1_AF488_crosstalk",
    "T1_Cy3_crosstalk",
    "T1_Cy5_crosstalk",
    "T1_Cy7_crosstalk",
    "T2_AF405_crosstalk",
    "T2_AF488_crosstalk",
    "T2_Cy3_crosstalk",
    "T2_Cy5_crosstalk",
    "T2_Cy7_crosstalk",
    "T3_AF405_crosstalk",
    "T3_AF488_crosstalk",
    "T3_Cy3_crosstalk",
    "T3_Cy5_crosstalk",
    "T3_Cy7_crosstalk"
]

# Read all Excel files and store the data
excel_data = {}
for file_name in file_order:
    file_path = os.path.join(folder_path, f"{file_name}.xlsx")
    try:
        # Read Excel file (without header)
        df = pd.read_excel(file_path, header=None)
        excel_data[file_name] = df
        print(f"Successfully read file: {file_name}.xlsx")
    except Exception as e:
        print(f"Error reading file {file_name}.xlsx: {str(e)}")

# Check if all files were successfully read
missing_files = [file for file in file_order if file not in excel_data]
if missing_files:
    print("\nThe following files are missing, cannot continue processing:")
    for file in missing_files:
        print(f"- {file}.xlsx")
else:
    # Get the shape of the first file to ensure all files have consistent dimensions
    first_file = file_order[0]
    rows, cols = excel_data[first_file].shape
    
    # Check if all files have the same dimensions
    size_mismatch = False
    for file_name in file_order[1:]:
        if excel_data[file_name].shape != (rows, cols):
            print(f"\nFile {file_name}.xlsx has mismatched dimensions with other files")
            size_mismatch = True
    
    if size_mismatch:
        print("Please ensure all Excel files have the same number of rows and columns")
    else:
        # Create an empty DataFrame with the same dimensions as the original files
        result_df = pd.DataFrame(index=range(rows), columns=range(cols))
        
        # Iterate through each pixel position (i,j) while maintaining original coordinates
        for i in range(rows):
            for j in range(cols):
                # Extract 15 values from corresponding positions to form a list
                values = []
                for file_name in file_order:
                    # Get cell value, ensuring it's a numeric type
                    value = excel_data[file_name].iloc[i, j]
                    values.append(float(value) if pd.notna(value) else 0.0)
                
                # Convert list to string without brackets (e.g., "1, 0, 0, ...")
                # To keep integer format, use: ", ".join(map(str, map(int, values)))
                matrix_str = ", ".join(map(str, values))
                # Place result in the same cell position as original
                result_df.iloc[i, j] = matrix_str
        
        # Save results to a new Excel file
        output_path = os.path.join(folder_path, "fluoro_changing_data.xlsx")
        result_df.to_excel(output_path, index=False, header=False)
        
        print(f"\nProcessing completed! Generated pure number strings maintain original Excel position distribution")
        print(f"Results saved to: {output_path}")

Successfully read file: T1_AF405_crosstalk.xlsx
Successfully read file: T1_AF488_crosstalk.xlsx
Successfully read file: T1_Cy3_crosstalk.xlsx
Successfully read file: T1_Cy5_crosstalk.xlsx
Successfully read file: T1_Cy7_crosstalk.xlsx
Successfully read file: T2_AF405_crosstalk.xlsx
Successfully read file: T2_AF488_crosstalk.xlsx
Successfully read file: T2_Cy3_crosstalk.xlsx
Successfully read file: T2_Cy5_crosstalk.xlsx
Successfully read file: T2_Cy7_crosstalk.xlsx
Successfully read file: T3_AF405_crosstalk.xlsx
Successfully read file: T3_AF488_crosstalk.xlsx
Successfully read file: T3_Cy3_crosstalk.xlsx
Successfully read file: T3_Cy5_crosstalk.xlsx
Successfully read file: T3_Cy7_crosstalk.xlsx

Processing completed! Generated pure number strings maintain original Excel position distribution
Results saved to: D://decoding simulation\fluoro_changing_data.xlsx


## Step 3. Linear unmixing

The fundamental concept of linear unmixing is based on the following formula:

 `array_m = C1*array_1 + C2*array_2 + …… + Cn*array_n`

- `array_m`：Color-changing information of each pixel.
- `array_n`: Fluoro-changing barcodes.
- `contribution factor (C)`: Contribution of array_n in making up array_m, which also corresponds to the relative fluorescence intensity of the recontrasted image.

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import mean_squared_error
from scipy.optimize import nnls
import openpyxl

# Read the Excel file (raw_data.xlsx)
input_df = pd.read_excel('D://decoding simulation//fluoro_changing_data.xlsx', header=None)

# color-changing barcode
arrays = [
  np.array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0]), #Pritenin 1
  np.array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0]), #Pritenin 2
  np.array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1]), #Pritenin 3
  np.array([0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0]), #Pritenin 4
  np.array([0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]), #Pritenin 5
  np.array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]), #Pritenin 6
  np.array([0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0]), #Pritenin 7
  np.array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0]), #Pritenin 8
  np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0]), #Pritenin 9
]

def find_combinations(array_m, arrays, max_arrays=5):
    best_combination = None
    best_constants = None
    best_score = float('inf')

    for n in range(1, max_arrays + 1):
        for combo in combinations(enumerate(arrays), n):
            indices, selected_arrays = zip(*combo)
            
            A = np.array(selected_arrays).T
            b = array_m
            try:
                constants, _ = nnls(A, b)
                predicted = np.dot(A, constants)
                score = mean_squared_error(array_m, predicted)
                
                if score < best_score:
                    best_score = score
                    best_combination = indices
                    best_constants = constants
            except ValueError:
                continue

    return best_combination, best_constants, best_score

# Creat a new Excel file to store results
workbook = openpyxl.Workbook()

# Define the sheet with array and protein name
sheet_names = ['Protein 1', 'Protein 2', 'Protein 3', 'Protein 4', 'Protein 5', 'Protein 6', 'Protein 7', 'Protein 8', 'Protein 9', 
               'Match Score']

# Create an empty DataFrame for each sheet and fill it with 0.0
results = {name: pd.DataFrame(0.0, index=input_df.index, columns=input_df.columns, dtype=float) for name in sheet_names}

# Calculate the optimal combination for each cell
for row in range(input_df.shape[0]):
    for col in range(input_df.shape[1]):
        cell_value = input_df.iloc[row, col]
        if pd.notna(cell_value):  # Check if the cell is empty
            array_m = np.array([float(x) for x in str(cell_value).split(',')])  # Convert the cell value to an array
            if len(array_m) == len(arrays[0]):  # Ensure length matches
                combination, constants, score = find_combinations(array_m, arrays)
                
                if score <= 20:
                    for i, const in zip(combination, constants):
                        results[sheet_names[i]].iloc[row, col] = const
                
                results['Match Score'].iloc[row, col] = score
            else:
                print(f"Warning: Mismatch in array length at row {row+1}, column {col+1}")

# Write the results to Excel file
with pd.ExcelWriter('D://decoding simulation//contribution_factor.xlsx', engine='openpyxl') as writer:
    for sheet_name, df in results.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

print("Results have been written to output_results.xlsx")

## Step 4: Image display

- The contribution factor decoded by linear unmixing can be displayed as a grayscale image (.tif) using the following code.
- To assign the grayscale image with pseudo color: Open zen 3.11 software → Dimensions → Channels.
- To merge different protein images: Open zen 3.11 software → Processing → Method → Add channes.

In [ ]:
import openpyxl
from PIL import Image
import numpy as np

def excel_to_tiff(excel_path, output_dir):
    # Open the Excel file
    workbook = openpyxl.load_workbook(excel_path)

    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]

        # Get the dimensions of the Excel sheet
        max_row = sheet.max_row
        max_col = sheet.max_column

        # Create a numpy array to store grayscale values, with data type np.uint8 (8-bit)
        image_array = np.zeros((max_row, max_col), dtype=np.uint8)

        # Read values from the Excel sheet and populate the array
        for row in range(1, max_row + 1):
            for col in range(1, max_col + 1):
                cell_value = sheet.cell(row=row, column=col).value
                if cell_value is not None:
                    # Ensure the value is within the range of an 8-bit unsigned integer (0-255)
                    cell_value = max(0, min(int(cell_value), 255))
                    image_array[row - 1, col - 1] = cell_value

        # Create an image with 8-bit mode ('L' for grayscale)
        image = Image.fromarray(image_array, mode='L')

        # Construct the output file path, naming it after the sheet name
        tiff_path = f"{output_dir}/{sheet_name}.tif"

        # Save the image as a TIFF file
        image.save(tiff_path, format='TIFF')
        print(f"TIFF image has been saved to {tiff_path}")

# Specify the input Excel file path and output directory path
excel_file_path = r"D://decoding simulation//contribution_factor.xlsx"  # Replace with your Excel file path
output_directory = r"D://decoding simulation//output_images"  # Replace with the directory path where you want to save the TIFF files

# Call the function
excel_to_tiff(excel_file_path, output_directory)